## Semana 2 Día 2

¡Nuestro primer proyecto de Agentic Framework!

Prepárate para algo increíblemente fácil.

Vamos a crear un sistema de agentes sencillo para generar correos electrónicos de contacto de ventas en frío:
1. Flujo de trabajo del agente
2. Uso de herramientas para llamar a funciones
3. Colaboración de los agentes mediante herramientas y transferencias

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import csv
import json
from datetime import datetime

In [2]:
load_dotenv(override=True)

True

In [3]:
names = lambda obj, txt = "": print([name for name in dir(obj) if (name[0] != "_") and (txt in name)])

## Paso 1: Flujo de trabajo del agente

Definimos tres tipos de Persona mediante los tres `system_prompt`:

In [4]:
instructions1 = "Eres un agente de ventas que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío profesionales y serios."

instructions2 = "Eres un agente de ventas con sentido del humor y atractivo \
    que trabaja para ComplAI, una empresa que ofrece una herramienta SaaS para \
    garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
    Redactas correos electrónicos en frío ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Eres un agente de ventas muy activo que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío concisos y directos."

El agente (`Agent`) tiene una un nombre e intrucciones (Persona) y un LLM.

In [5]:
sales_agent1 = Agent(
        name="Agente de ventas profesional",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Agente de ventas atractivo",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Agente de ventas ocpado",
        instructions=instructions3,
        model="gpt-4o-mini"
)

El método `run_streamed()` ejecuta en tiempo real la corrutina.`Runner.run_streamed()` es una función asíncrona y cualquier objeto creado a partir de esta función será una corrutina.

Cada evento del objeto `result.stream_events()` es una corrutina.

Con `end=""` consigo empalmar el texto a manera de stream.

Utilizamos el primer tipo de agente de ventas para observar el contenido el email a manera de streaming.

In [6]:

result = Runner.run_streamed(sales_agent1, input="Escribe un correo electrónico de ventas en frío")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Asunto: Asegure su Cumplimiento SOC 2 de Manera Eficiente

Estimado/a [Nombre del destinatario]:

Espero que este mensaje le encuentre bien. Mi nombre es [Tu Nombre] y soy representante de ComplAI, una solución SaaS innovadora diseñada para ayudar a empresas como la suya a garantizar el cumplimiento de SOC 2 y optimizar la preparación para auditorías.

En un entorno regulatorio creciente y complejo, el cumplimiento de SOC 2 no solo es crucial para la confianza de sus clientes, sino también para la sostenibilidad de su negocio. Nuestra herramienta, impulsada por inteligencia artificial, simplifica los procesos, reduce el riesgo de auditorías fallidas y proporciona análisis detallados que permiten una gestión proactiva de la conformidad.

Algunas de las características clave de ComplAI son:

- **Automatización de Procesos**: Minimiza el tiempo y los recursos dedicados a la gestión del cumplimiento.
- **Auditorías Efectivas**: Prepara a su equipo con herramientas para responder a las audi

`asyncio.gather()` Usará un bucle de eventos donde ejecutará un evento cuando se necesite y se pause en caso contrario. En este caso tenemos tres eventos que corresponde a tres corrutinas de los tres agentes.

In [7]:
message = "Escribe un correo electrónico de ventas en frío"

with trace("Correos electrónicos fríos en paralelo"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Asunto: Mejore su Cumplimiento de SOC2 con ComplAI

Estimado/a [Nombre del destinatario],

Espero que este mensaje le encuentre bien. Mi nombre es [Tu Nombre] y soy parte del equipo de ComplAI, donde ofrecemos soluciones innovadoras para ayudar a las empresas a garantizar el cumplimiento de SOC2 de manera eficiente y efectiva.

En el entorno empresarial actual, cumplir con las normativas y estándares de seguridad es crucial, no solo para proteger la información de sus clientes, sino también para mantener la confianza en su marca. Nuestra herramienta SaaS, impulsada por inteligencia artificial, simplifica y optimiza el proceso de auditoría, permitiéndole prepararse con antelación y reducir significativamente el tiempo y los recursos dedicados a esta tarea.

Algunas de las características clave de ComplAI incluyen:

- **Automatización de Procesos:** Minimice los esfuerzos manuales con flujos de trabajo automatizados que simplifican la recolección de documentación y evidencia necesaria.
-

Ahora creamos nuestro flujo de trabajo o workflow a LLMs.

![alt text](image.png)

El siguiente agente, va a escoger entre los tres tipos de correos, el mejor.

Deinimos su Persona (`name e instructions`) y el LLM.

In [8]:
sales_picker = Agent(
    name="sales_picker",
    instructions="Elige el mejor correo electrónico de ventas en frío entre las opciones disponibles. \
        Imagina que eres un cliente y elige el que probablemente te responda. \
        No des explicaciones; responde solo con el correo electrónico seleccionado.",
    model="gpt-4o-mini"
)

## Parte 2: Uso de herramientas

Ahora añadiremos una herramienta.

Recuerda todo el código JSON repetitivo y la función `handle_tool_calls()` con la lógica `if`.

In [9]:
sales_agent1 = Agent(
        name="Agente de ventas profesional",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Agente de ventas atractivo",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Agente de ventas ocpado",
        instructions=instructions3,
        model="gpt-4o-mini"
)

## Pasos 2 y 3: Interacción con herramientas y agentes

Con el método `.as_tool()` se convierte al agente `sales_agent1` en una herramienta con su correspondiente JSON schema.

In [10]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Escribe un email de ventas en frío")
tool1

FunctionTool(name='sales_agent1', description='Escribe un email de ventas en frío', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000021F29FE4400>, strict_json_schema=True)

### Ahora podemos reunir todas las herramientas:

Una herramienta para cada uno de nuestros tres agentes de redacción de correos electrónicos

Y una herramienta para nuestra función de envío de correos electrónicos

In [11]:
description = "Escribe un correo electrónico de ventas en frío"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

### Las transferencias (handoffs) representan una forma en que un agente puede delegar en otro agente, transfiriéndole el control.

Las transferencias y los agentes como herramientas son similares:

En ambos casos, un agente puede colaborar con otro.

Con las herramientas, el control se queda en un agente.

Con las transferencias (handoffs), el control se transfiere o se delega a otro agente.

En el caso de envío de correo electrónico, esta tarea la podemos dividir en: escribir el Asunto y el redactar su contenido. Para ello creamos dos agentes para encargarse de estas dos subtareas.

In [12]:
subject_instructions = "Puedes escribir un asunto para un correo electrónico de ventas en frío. \
    Se te proporciona un mensaje y necesitas escribir un asunto para un correo electrónico que probablemente obtenga respuesta."

html_instructions = "Puedes convertir un cuerpo de correo electrónico de texto a un cuerpo de correo electrónico HTML. \
    Se te proporciona un cuerpo de correo electrónico de texto que puede tener algún markdown \
    y necesitas convertirlo a un cuerpo de correo electrónico HTML con un diseño simple, claro y atractivo."

subject_writer = Agent(name="Escritor de asunto de correo electrónico", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", 
                                      tool_description="Escribe un asunto para un correo electrónico de ventas en frío")

html_converter = Agent(name="Conversor de cuerpo de correo electrónico HTML", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",
                                   tool_description="Convierte un cuerpo de correo electrónico de texto a un cuerpo de correo electrónico HTML")


subject_witer y html_converter van a ser tools utilizando el metodo `as_tool()`.

Ahora la nueva herramienta de envio de correos tiene dos argumentos: `subject: str y html_body: str`.

In [13]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envía un correo electrónico con el asunto y el cuerpo HTML a todos los clientes potenciales de ventas """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("100289351@alumnos.uc3m.es")  # Cambiar a tu remitente verificado
    to_email = To("actmartz@gmail.com")  # Cambiar a sureceptor
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

Ahora tenemos tres herramientas a disposición que van al stack o lista de herramientas `tools`.

In [14]:
tools = [subject_tool, html_tool, send_html_email]

In [15]:
tools

[FunctionTool(name='subject_writer', description='Escribe un asunto para un correo electrónico de ventas en frío', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000021F29FE51C0>, strict_json_schema=True),
 FunctionTool(name='html_converter', description='Convierte un cuerpo de correo electrónico de texto a un cuerpo de correo electrónico HTML', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000021F29FE4E00>, strict_json_schema=True),
 FunctionTool(name='send_html_email', description='Envía un correo electrónic

Creamos un nuevo agente al cual se va a hacer el handoff.

In [16]:
instructions ="Eres un formateador y remitente de correos electrónicos. \
    Recibes el cuerpo de un correo electrónico para enviarlo. \
    Primero usas la herramienta subject_writer para escribir un asunto para el correo electrónico, \
    luego usas la herramienta html_converter para convertir el cuerpo a HTML. \
    Finalmente, usas la herramienta send_html_email para enviar el correo electrónico con el asunto y el cuerpo HTML."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convierte un email a HTML y lo envía")


Con `handoff_description` se otorga una transferencia de trabajo (creación del correo a HTML) usando el stack recientemente creado `tools`.

### Ahora tenemos 3 herramientas y 1 transferencia

Todas las herramientas se van a ejecutar como corrutinas. Estas tres ultimas van a ser las herramientas del gerente de ventas.

In [17]:
tools = [tool1, tool2, tool3] # Creación de contenido de tres tipos de agentes
handoffs = [emailer_agent]# Redacta el correo en formato HTML
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Escribe un correo electrónico de ventas en frío', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000021F29FE4AE0>, strict_json_schema=True), FunctionTool(name='sales_agent2', description='Escribe un correo electrónico de ventas en frío', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000021F29F4BCE0>, strict_json_schema=True), FunctionTool(name='sales_agent3', description='Escribe un correo electrónico de ventas en frío', params_json_schema={'properties': {'input': {'

In [18]:
sales_manager_instructions = "Eres un gerente de ventas que trabaja para ComplAI. Utilizas las herramientas que se te proporcionan para generar correos electrónicos de ventas en frío. \
Nunca generas correos electrónicos de ventas tú mismo; siempre utilizas las herramientas. \
Pruebas las 3 herramientas del agente de ventas al menos una vez antes de elegir la mejor. \
Puedes usar las herramientas múltiples veces si no estás satisfecho con los resultados del primer intento. \
Seleccionas el mejor correo electrónico usando tu propio criterio sobre cuál será más efectivo. \
Después de elegir el correo electrónico, transfieres al agente Email Manager para formatear y enviar el correo."


sales_manager = Agent(
    name="Manager de ventas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Envía un correo electrónico de ventas en frío dirigido a 'Estimado director ejecutivo'"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

EL Manager de ventas (`sales_manager`) elige el correo de los tres agentes anteriores y transfiere la creación del correo final en formato HTML al agente `emailer_agent`.

Por consiguiente este ejercicio nos encontramos con un caso de **Hierarchical Agents**.

### Recuerda revisar el seguimiento

https://platform.openai.com/traces

¡Y luego revisa tu correo electrónico!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ejercicio</h2>
<span style="color:#ff7800;">¿Puedes identificar los patrones de diseño de Agentic que se han usado aquí?<br/>
¿Cuál es la línea que cambió esto de ser un "flujo de trabajo" de Agentic a un "agente" según la definición de Anthropic?<br/>
¡Intenta agregar más herramientas y agentes! Podrías tener herramientas que gestionen la combinación de correspondencia para enviar a una lista.<br/><br/>
RETO DIFÍCIL: Investiga cómo puedes hacer que SendGrid llame a un webhook de devolución de llamada cuando un usuario responde a un correo electrónico.
Luego, haz que el SDR responda para continuar la conversación. Esto puede requerir algo de programación de ambiente. 😂
</span>
        </td>
    </tr>
</table>


# ¿Puedes identificar los patrones de diseño de Agentic que se han usado aquí?
### El `sales_manager` es un agente Orquestador-Sintetizador:

```python
sales_manager = Agent(
    name="Manager de ventas",
    instructions=sales_manager_instructions,
    tools=tools,  # [tool1, tool2, tool3] - 3 agentes en paralelo
    handoffs=handoffs,  # [emailer_agent]
    model="gpt-4o-mini")
```

**Flujo:**
```
                    ┌─→ [sales_agent1] → email profesional
                    │
sales_manager ──────┼─→ [sales_agent2] → email atractivo
(Orquestador)       │
                    └─→ [sales_agent3] → email directo
                              ↓
                    [Sintetizador: elige el mejor]
                              ↓
                    [Handoff → emailer_agent]
```

**Aquí SÍ hay:**
1. ✅ **Orquestación**: Llama a 3 agentes en paralelo
2. ✅ **Sintetización**: Evalúa y selecciona el mejor resultado
3. ✅ **Delegación**: Transfiere al `emailer_agent` para finalizar

---

## Visualización del Sistema Completo

```
┌─────────────────────────────────────────────────┐
│          SALES MANAGER                          │
│     (Orquestador-Sintetizador)                  │
│                                                 │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐     │
│  │ Agent 1  │  │ Agent 2  │  │ Agent 3  │     │
│  │Professional│ │ Engaging │  │ Direct   │     │
│  └─────┬────┘  └─────┬────┘  └─────┬────┘     │
│        └──────────────┼─────────────┘          │
│                       ↓                         │
│           [Síntesis: elige mejor]               │
│                       ↓                         │
│                  [Handoff]                      │
└───────────────────────┼─────────────────────────┘
                        ↓
┌───────────────────────┼─────────────────────────┐
│          EMAILER AGENT                          │
│          (Pipeline)                             │
│                                                 │
│         [subject_writer]                        │
│                ↓                                │
│         [html_converter]                        │
│                ↓                                │
│         [send_html_email]                       │
│                ↓                                │
│            [Output]                             │
└─────────────────────────────────────────────────┘
```

---

### `emailer_agent` es un **Especialista con Pipeline Interno**

- **Especialista**: Tiene un dominio específico (formatear y enviar emails)
- **Pipeline**: Ejecuta una secuencia de transformaciones
- **Worker Agent**: Recibe trabajo delegado (handoff) y lo ejecuta

# ¿Cuál es la línea que cambió esto de ser un "flujo de trabajo" de Agentic a un "agente" según la definición de Anthropic?

**La línea crucial que transformó esto de "workflow" a "agente":**

```python
sales_manager = Agent(
    name="Manager de ventas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,  # ← ESTA LÍNEA
    model="gpt-4o-mini")
```

### ¿Por qué?

**Workflow estático:**
- Secuencia predefinida de pasos
- El flujo está hardcodeado por el programador
- No hay decisiones dinámicas basadas en contexto

**Agente verdadero (con `handoffs`):**
- El LLM **decide** cuándo transferir control
- El agente **razona** sobre qué herramienta usar
- El flujo es **dinámico** y adaptativo al contexto
- Hay **autonomía** en la toma de decisiones

Según la definición de Anthropic, un agente tiene:
1. Ciclos iterativos (tool loops)
2. Razonamiento dinámico
3. Toma de decisiones autónoma
4. Capacidad de auto-corrección




# ¡Intenta agregar más herramientas y agentes! Podrías tener herramientas que gestionen la combinación de correspondencia para enviar a una lista.

In [ ]:
load_dotenv(override=True)

# ============================================================================
# HERRAMIENTAS DE GESTIÓN DE LISTAS
# ============================================================================

@function_tool
def load_prospects_from_csv(file_path: str) -> Dict[str, any]:
    """Carga una lista de prospectos desde un archivo CSV.
    El CSV debe tener columnas: email, nombre, empresa, cargo"""
    try:
        prospects = []
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                prospects.append({
                    'email': row.get('email', ''),
                    'nombre': row.get('nombre', ''),
                    'empresa': row.get('empresa', ''),
                    'cargo': row.get('cargo', '')
                })
        return {
            "status": "success",
            "count": len(prospects),
            "prospects": prospects
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
@function_tool
def create_prospect_list(prospects_json: str) -> Dict[str, any]:
    """Crea una lista de prospectos desde un JSON string.
    Formato: [{"email": "...", "nombre": "...", "empresa": "...", "cargo": "..."}]"""
    try:
        prospects = json.loads(prospects_json)
        return {
            "status": "success",
            "count": len(prospects),
            "prospects": prospects
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
@function_tool
def filter_prospects_by_criteria(prospects_json: str, criteria: str) -> Dict[str, any]:
    """Filtra prospectos según un criterio específico (ej: cargo='CEO', empresa='Tech')"""
    try:
        prospects = json.loads(prospects_json)
        # Filtrado simple por palabra clave
        filtered = [p for p in prospects if criteria.lower() in str(p).lower()]
        return {
            "status": "success",
            "original_count": len(prospects),
            "filtered_count": len(filtered),
            "prospects": filtered
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
# ============================================================================
# HERRAMIENTAS DE PERSONALIZACIÓN (MAIL MERGE)
# ============================================================================

@function_tool
def personalize_email_template(template: str, prospect_data: str) -> Dict[str, str]:
    """Personaliza una plantilla de email con datos del prospecto.
    Usa marcadores: {nombre}, {empresa}, {cargo}"""
    try:
        prospect = json.loads(prospect_data)
        personalized = template.format(
            nombre=prospect.get('nombre', '[Nombre]'),
            empresa=prospect.get('empresa', '[Empresa]'),
            cargo=prospect.get('cargo', '[Cargo]')
        )
        return {
            "status": "success",
            "personalized_email": personalized
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
@function_tool
def generate_personalized_emails(email_body: str, prospects_json: str) -> Dict[str, any]:
    """Genera múltiples emails personalizados para una lista de prospectos"""
    try:
        prospects = json.loads(prospects_json)
        personalized_emails = []
        
        for prospect in prospects:
            try:
                personalized = email_body.format(
                    nombre=prospect.get('nombre', '[Nombre]'),
                    empresa=prospect.get('empresa', '[Empresa]'),
                    cargo=prospect.get('cargo', '[Cargo]')
                )
                personalized_emails.append({
                    'email': prospect['email'],
                    'content': personalized,
                    'prospect': prospect
                })
            except KeyError:
                continue
        
        return {
            "status": "success",
            "count": len(personalized_emails),
            "emails": personalized_emails
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
# ============================================================================
# HERRAMIENTAS DE ENVÍO MASIVO
# ============================================================================

@function_tool
def send_batch_emails(subject: str, emails_json: str, from_email: str = "100289351@alumnos.uc3m.es") -> Dict[str, any]:
    """Envía un lote de emails personalizados a múltiples destinatarios"""
    try:
        sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
        emails_data = json.loads(emails_json)
        
        results = []
        for email_item in emails_data:
            try:
                mail = Mail(
                    from_email=Email(from_email),
                    to_emails=To(email_item['email']),
                    subject=subject,
                    html_content=Content("text/html", email_item['content'])
                ).get()
                
                response = sg.client.mail.send.post(request_body=mail)
                results.append({
                    'email': email_item['email'],
                    'status': 'sent',
                    'status_code': response.status_code
                })
            except Exception as e:
                results.append({
                    'email': email_item['email'],
                    'status': 'failed',
                    'error': str(e)
                })
        
        successful = len([r for r in results if r['status'] == 'sent'])
        return {
            "status": "completed",
            "total": len(results),
            "successful": successful,
            "failed": len(results) - successful,
            "results": results
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
@function_tool
def schedule_email_campaign(campaign_name: str, send_time: str, emails_json: str) -> Dict[str, str]:
    """Programa una campaña de emails para ser enviada en un momento específico"""
    # Esta sería una implementación básica - en producción usarías un sistema de colas
    return {
        "status": "scheduled",
        "campaign_name": campaign_name,
        "send_time": send_time,
        "emails_count": len(json.loads(emails_json)),
        "message": f"Campaña '{campaign_name}' programada para {send_time}"
    }

In [ ]:
# ============================================================================
# HERRAMIENTAS DE ANÁLISIS Y SEGUIMIENTO
# ============================================================================

@function_tool
def track_email_campaign(campaign_id: str) -> Dict[str, any]:
    """Simula el seguimiento de métricas de una campaña de email"""
    # En producción, esto consultaría una base de datos real
    return {
        "campaign_id": campaign_id,
        "sent": 150,
        "opened": 45,
        "clicked": 12,
        "replied": 3,
        "open_rate": "30%",
        "click_rate": "8%",
        "reply_rate": "2%"
    }

In [ ]:
@function_tool
def generate_campaign_report(campaign_data: str) -> Dict[str, str]:
    """Genera un reporte de campaña con métricas y recomendaciones"""
    try:
        data = json.loads(campaign_data)
        report = f"""
        REPORTE DE CAMPAÑA
        ==================
        Total enviados: {data.get('sent', 0)}
        Tasa de apertura: {data.get('open_rate', 'N/A')}
        Tasa de clic: {data.get('click_rate', 'N/A')}
        Tasa de respuesta: {data.get('reply_rate', 'N/A')}
        
        Estado: {'Excelente' if float(data.get('reply_rate', '0%').strip('%')) > 5 else 'Necesita optimización'}
        """
        return {"status": "success", "report": report}
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
# ============================================================================
# AGENTES ORIGINALES
# ============================================================================

instructions1 = "Eres un agente de ventas que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío profesionales y serios."

instructions2 = "Eres un agente de ventas con sentido del humor y atractivo \
    que trabaja para ComplAI, una empresa que ofrece una herramienta SaaS para \
    garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
    Redactas correos electrónicos en frío ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Eres un agente de ventas muy activo que trabaja para ComplAI, \
    una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2\
    y prepararse para auditorías, impulsada por IA. Redactas correos electrónicos en frío concisos y directos."

sales_agent1 = Agent(
    name="Agente de ventas profesional",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Agente de ventas atractivo",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Agente de ventas ocupado",
    instructions=instructions3,
    model="gpt-4o-mini"
)

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", 
                              tool_description="Escribe un email de ventas en frío profesional")
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", 
                              tool_description="Escribe un email de ventas en frío atractivo")
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", 
                              tool_description="Escribe un email de ventas en frío conciso")

In [ ]:
# ============================================================================
# NUEVOS AGENTES ESPECIALIZADOS
# ============================================================================

# Agente de gestión de listas
list_manager_instructions = """Eres un experto en gestión de listas de prospectos.
Puedes cargar listas desde archivos CSV, crear listas manualmente, y filtrar prospectos
según criterios específicos. Siempre proporcionas conteos y resúmenes de las operaciones."""

list_manager = Agent(
    name="List Manager",
    instructions=list_manager_instructions,
    tools=[load_prospects_from_csv, create_prospect_list, filter_prospects_by_criteria],
    model="gpt-4o-mini",
    handoff_description="Gestiona listas de prospectos: carga, crea y filtra"
)

# Agente de personalización
personalization_instructions = """Eres un experto en personalización de emails.
Tomas plantillas de email y las personalizas con datos de prospectos.
Siempre te aseguras de que los marcadores {nombre}, {empresa}, {cargo} estén presentes
en las plantillas antes de personalizar."""

personalization_agent = Agent(
    name="Personalization Agent",
    instructions=personalization_instructions,
    tools=[personalize_email_template, generate_personalized_emails],
    model="gpt-4o-mini",
    handoff_description="Personaliza emails con datos de prospectos"
)

# Agente de envío masivo
batch_sender_instructions = """Eres un especialista en envío masivo de emails.
Gestionas el envío de múltiples emails personalizados, programas campañas,
y te aseguras de que todos los envíos se realicen correctamente."""

batch_sender = Agent(
    name="Batch Sender",
    instructions=batch_sender_instructions,
    tools=[send_batch_emails, schedule_email_campaign],
    model="gpt-4o-mini",
    handoff_description="Envía emails en lote y programa campañas"
)

# Agente de análisis
analytics_instructions = """Eres un analista de campañas de email marketing.
Proporcionas métricas, seguimiento y reportes de campañas.
Das insights accionables basados en los datos."""

analytics_agent = Agent(
    name="Analytics Agent",
    instructions=analytics_instructions,
    tools=[track_email_campaign, generate_campaign_report],
    model="gpt-4o-mini",
    handoff_description="Analiza y reporta métricas de campañas"
)

# Agentes auxiliares originales
subject_instructions = "Puedes escribir un asunto para un correo electrónico de ventas en frío. \
    Se te proporciona un mensaje y necesitas escribir un asunto para un correo electrónico que probablemente obtenga respuesta."

html_instructions = "Puedes convertir un cuerpo de correo electrónico de texto a un cuerpo de correo electrónico HTML. \
    Se te proporciona un cuerpo de correo electrónico de texto que puede tener algún markdown \
    y necesitas convertirlo a un cuerpo de correo electrónico HTML con un diseño simple, claro y atractivo."

subject_writer = Agent(
    name="Escritor de asunto de correo electrónico",
    instructions=subject_instructions,
    model="gpt-4o-mini"
)

html_converter = Agent(
    name="Conversor de cuerpo de correo electrónico HTML",
    instructions=html_instructions,
    model="gpt-4o-mini"
)

subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Escribe un asunto para un correo electrónico de ventas en frío"
)

html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convierte un cuerpo de correo electrónico de texto a HTML"
)

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Envía un correo electrónico individual con el asunto y el cuerpo HTML"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("100289351@alumnos.uc3m.es")
    to_email = To("actmartz@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

# Agente de email mejorado
emailer_instructions = """Eres un formateador y remitente de correos electrónicos.
Recibes el cuerpo de un correo electrónico para enviarlo.
Primero usas la herramienta subject_writer para escribir un asunto para el correo electrónico,
luego usas la herramienta html_converter para convertir el cuerpo a HTML.
Finalmente, usas la herramienta send_html_email para enviar el correo electrónico con el asunto y el cuerpo HTML."""

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=[subject_tool, html_tool, send_html_email],
    model="gpt-4o-mini",
    handoff_description="Convierte un email a HTML y lo envía"
)

In [ ]:
# ============================================================================
# GERENTE DE VENTAS MEJORADO
# ============================================================================

sales_manager_instructions = """Eres un gerente de ventas avanzado que trabaja para ComplAI.
Coordinas todo el proceso de outreach de ventas, desde la creación de contenido hasta el envío masivo.

Tu flujo de trabajo típico:
1. Para campañas individuales: generas emails con las 3 herramientas de agentes de ventas, eliges el mejor, 
   y transfieres al Email Manager para formatearlo y enviarlo.

2. Para campañas masivas: 
   - Primero transfieres al List Manager para obtener la lista de prospectos
   - Luego generas una plantilla de email con marcadores {nombre}, {empresa}, {cargo}
   - Transfieres al Personalization Agent para personalizar los emails
   - Transfieres al Batch Sender para enviar los emails en lote
   - Opcionalmente, transfieres al Analytics Agent para ver métricas

Siempre utilizas las herramientas y transferencias apropiadas según el tipo de campaña solicitada."""

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=[tool1, tool2, tool3],
    handoffs=[
        emailer_agent,
        list_manager,
        personalization_agent,
        batch_sender,
        analytics_agent
    ],
    model="gpt-4o-mini"
)

In [ ]:
# ============================================================================
# EJEMPLOS DE USO
# ============================================================================

async def ejemplo_campana_individual():
    """Ejemplo: Enviar un email individual"""
    message = "Envía un correo electrónico de ventas en frío dirigido a 'Estimado director ejecutivo'"
    
    with trace("Campaña Individual"):
        result = await Runner.run(sales_manager, message)
    
    print("\n=== RESULTADO CAMPAÑA INDIVIDUAL ===")
    print(result.final_output)


async def ejemplo_campana_masiva():
    """Ejemplo: Enviar una campaña masiva con mail merge"""
    
    # Primero crear una lista de prospectos de ejemplo
    prospects = [
        {"email": "ceo1@empresa.com", "nombre": "María González", "empresa": "TechCorp", "cargo": "CEO"},
        {"email": "cto@startup.com", "nombre": "Juan Pérez", "empresa": "InnovaAI", "cargo": "CTO"},
        {"email": "director@company.com", "nombre": "Ana Martínez", "empresa": "SecureData", "cargo": "Directora de TI"}
    ]
    
    message = f"""Crea y envía una campaña de email masiva:
    1. Usa esta lista de prospectos: {json.dumps(prospects)}
    2. Genera una plantilla de email personalizada que use los marcadores {{nombre}}, {{empresa}}, {{cargo}}
    3. Personaliza y envía los emails a todos los prospectos
    4. Muéstrame un resumen de la campaña"""
    
    with trace("Campaña Masiva"):
        result = await Runner.run(sales_manager, message)
    
    print("\n=== RESULTADO CAMPAÑA MASIVA ===")
    print(result.final_output)


async def ejemplo_con_filtrado():
    """Ejemplo: Filtrar prospectos y enviar campaña segmentada"""
    
    prospects = [
        {"email": "ceo1@empresa.com", "nombre": "María", "empresa": "TechCorp", "cargo": "CEO"},
        {"email": "cfo@empresa.com", "nombre": "Pedro", "empresa": "TechCorp", "cargo": "CFO"},
        {"email": "cto@startup.com", "nombre": "Juan", "empresa": "InnovaAI", "cargo": "CTO"},
    ]
    
    message = f"""Realiza una campaña segmentada:
    1. Toma esta lista: {json.dumps(prospects)}
    2. Filtra solo los CEOs
    3. Crea un email específico para CEOs
    4. Envía la campaña solo a los CEOs filtrados"""
    
    with trace("Campaña Segmentada"):
        result = await Runner.run(sales_manager, message)
    
    print("\n=== RESULTADO CAMPAÑA SEGMENTADA ===")
    print(result.final_output)


# Para ejecutar los ejemplos:
if __name__ == "__main__":
    # Descomentar el ejemplo que quieras probar:
    
    # asyncio.run(ejemplo_campana_individual())
    # asyncio.run(ejemplo_campana_masiva())
    # asyncio.run(ejemplo_con_filtrado())
    
    pass